# 01 — Exploratory Data Analysis (EDA)
**Project:** GB Compound Hazards Risk Prioritization  
**Course:** Artificial Intelligence in Human Water Systems  
**Reference:** Tilloy et al. (2022), Earth System Dynamics, 13, 993–1020

---
## Objectives
- Load and inspect ERA5 precipitation and wind gust datasets
- Understand the spatial and temporal structure of the data
- Visualize the distribution of extreme values (p0005)
- Confirm the study area covers Great Britain


## 1. Imports

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully!')

## 2. Load Data

In [ ]:
# Paths to raw data
RAIN_PATH = '../data/raw/era5_precipitation/raindat_0919.nc'
WIND_PATH = '../data/raw/era5_wind/windat_0919.nc'

# Load datasets
rain_ds = xr.open_dataset(RAIN_PATH)
wind_ds = xr.open_dataset(WIND_PATH)

print('Rain dataset:')
print(rain_ds)
print('\nWind dataset:')
print(wind_ds)

## 3. Extract Extreme Values (p0005)

> **Note:** Variable `p0005` contains only pre-filtered extreme observations provided by the data source. This is consistent with the percentile-based threshold approach of Tilloy et al. (2022), where only the top extreme values are retained for clustering. Variable `p0001` contains the complementary non-extreme observations and will not be used in this analysis.

In [ ]:
# Extract extreme values only
rain_extreme = rain_ds['p0005']  # Extreme precipitation (m) — convert to mm below
wind_extreme = wind_ds['p0005']  # Extreme wind gust (m/s)

# Convert precipitation from metres to millimetres
rain_extreme = rain_extreme * 1000

print('Extreme precipitation (mm):')
print(f'  Shape: {rain_extreme.shape}')
print(f'  Time range: {rain_extreme.time.values[0]} to {rain_extreme.time.values[-1]}')
print(f'  Non-NaN values: {int(rain_extreme.count().values):,}')
print(f'  Min: {float(rain_extreme.min().values):.4f} mm')
print(f'  Max: {float(rain_extreme.max().values):.4f} mm')

print('\nExtreme wind gust (m/s):')
print(f'  Shape: {wind_extreme.shape}')
print(f'  Time range: {wind_extreme.time.values[0]} to {wind_extreme.time.values[-1]}')
print(f'  Non-NaN values: {int(wind_extreme.count().values):,}')
print(f'  Min: {float(wind_extreme.min().values):.4f} m/s')
print(f'  Max: {float(wind_extreme.max().values):.4f} m/s')

## 4. Spatial Coverage — Study Area

In [ ]:
lats = rain_ds.latitude.values
lons = rain_ds.longitude.values

print(f'Latitude range:  {lats.min():.2f}°N to {lats.max():.2f}°N')
print(f'Longitude range: {lons.min():.2f}°E to {lons.max():.2f}°E')
print(f'Grid resolution: {abs(lats[1]-lats[0]):.2f}° x {abs(lons[1]-lons[0]):.2f}°')
print(f'Grid size: {len(lats)} latitudes x {len(lons)} longitudes = {len(lats)*len(lons)} cells')

## 5. Temporal Coverage

In [ ]:
rain_times = pd.to_datetime(rain_ds.time.values)
wind_times = pd.to_datetime(wind_ds.time.values)

print('Precipitation temporal coverage:')
print(f'  Start: {rain_times.min()}')
print(f'  End:   {rain_times.max()}')
print(f'  Total timesteps: {len(rain_times):,}')

print('\nWind temporal coverage:')
print(f'  Start: {wind_times.min()}')
print(f'  End:   {wind_times.max()}')
print(f'  Total timesteps: {len(wind_times):,}')

## 6. Visualize Spatial Distribution of Extreme Events

In [ ]:
# Count extreme events per grid cell
rain_count = rain_extreme.count(dim='time')
wind_count = wind_extreme.count(dim='time')

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Precipitation
im1 = axes[0].pcolormesh(
    rain_ds.longitude, rain_ds.latitude,
    rain_count, cmap='Blues'
)
axes[0].set_title('Count of Extreme Precipitation Events\n(2009–2019)', fontsize=13)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
plt.colorbar(im1, ax=axes[0], label='Number of extreme events')

# Wind
im2 = axes[1].pcolormesh(
    wind_ds.longitude, wind_ds.latitude,
    wind_count, cmap='Oranges'
)
axes[1].set_title('Count of Extreme Wind Gust Events\n(2009–2019)', fontsize=13)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
plt.colorbar(im2, ax=axes[1], label='Number of extreme events')

plt.tight_layout()
plt.savefig('../outputs/figures/01_spatial_distribution_extreme_events.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to outputs/figures/')

## 7. Temporal Distribution — Extreme Events per Year

In [ ]:
# Count extreme events per year across all grid cells
rain_df = rain_extreme.to_dataframe(name='precipitation').dropna()
wind_df = wind_extreme.to_dataframe(name='wind').dropna()

rain_per_year = rain_df.groupby(rain_df.index.get_level_values('time').year).size()
wind_per_year = wind_df.groupby(wind_df.index.get_level_values('time').year).size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(rain_per_year.index, rain_per_year.values, color='steelblue', edgecolor='white')
axes[0].set_title('Extreme Precipitation Events per Year', fontsize=13)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of extreme observations')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(wind_per_year.index, wind_per_year.values, color='darkorange', edgecolor='white')
axes[1].set_title('Extreme Wind Gust Events per Year', fontsize=13)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of extreme observations')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/figures/01_temporal_distribution_extreme_events.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to outputs/figures/')

## 8. Summary Statistics

In [ ]:
summary = pd.DataFrame({
    'Variable': ['Precipitation (p0005)', 'Wind Gust (p0005)'],
    'Units': ['mm', 'm/s'],
    'Extreme observations': [
        f"{int(rain_extreme.count().values):,}",
        f"{int(wind_extreme.count().values):,}"
    ],
    'Min': [
        f"{float(rain_extreme.min().values):.4f}",
        f"{float(wind_extreme.min().values):.4f}"
    ],
    'Max': [
        f"{float(rain_extreme.max().values):.4f}",
        f"{float(wind_extreme.max().values):.4f}"
    ],
    'Mean': [
        f"{float(rain_extreme.mean().values):.4f}",
        f"{float(wind_extreme.mean().values):.4f}"
    ]
})

print('=== EDA Summary ===')
print(summary.to_string(index=False))

## 9. Save Processed Data for Next Notebook

In [ ]:
# Save extreme dataframes for use in notebook 02
rain_df.to_parquet('../data/processed/rain_extreme_points.parquet')
wind_df.to_parquet('../data/processed/wind_extreme_points.parquet')

print('Saved:')
print('  data/processed/rain_extreme_points.parquet')
print('  data/processed/wind_extreme_points.parquet')
print(f'  Rain shape: {rain_df.shape}')
print(f'  Wind shape: {wind_df.shape}')